## Camada Gold
Este notebook é responsável pela consolidação final dos dados no pipeline ETL, construindo a camada Gold. O objetivo principal é gerar a tabela `gold_atendimentos_enriquecidos`, que é otimizada para o consumo de análises e dashboards de negócios.

* **Fontes de Dados (Inputs):**
* `workspace.silver.silver_atendimentos`: Tabela com os dados limpos das interações de atendimento.

* `workspace.raw.tb_info_clientes`: Tabela bruta contendo o cadastro e o perfil demográfico/fidelidade dos clientes.

* **Tabela Destino (Output):**
* `workspace.gold.gold_atendimentos_enriquecidos`.

* **Formato:** Delta.
* **Estratégia de Otimização:** Tabela particionada pela coluna `data_particao`.

* A construção da tabela Gold é feita através de um `LEFT JOIN` utilizando a chave `cliente_id`, conectando a base de atendimentos (Silver) com as informações dos clientes (Raw).

* **Colunas de Cliente Adicionadas:** Idade, estado de residência, quantidade de produtos adquiridos e categoria de fidelidade.

Para facilitar as consultas nas ferramentas de BI, a tabela compila colunas específicas (métricas e dimensões) para cada tipo de canal:

* **WhatsApp:** Número de origem, indicador de atendimento por bot e tempo de resposta do bot em segundos.

* **Telefone:** Duração da chamada e tempo de fila em segundos, protocolo e transferências.

* **Email:** Domínio, tamanho do corpo do e-mail em bytes, quantidade de anexos e tempo da primeira resposta.

* **Chat:** Browser, página de origem e satisfação pré-atendimento.

* A linhagem dos dados é mantida adicionando o `gold_processing_timestamp` via `CURRENT_TIMESTAMP()`, preservando também o timestamp original do processamento da camada Silver (`silver_processing_timestamp`).


In [0]:
# criação do Schema Gold

spark.sql("""
    CREATE DATABASE IF NOT EXISTS workspace.gold
    COMMENT 'Schema da Camada Gold'
""")

print("✓ Schema 'workspace.gold' criado/verificado com sucesso!")

In [0]:
# Criação da tabela Gold: atendimentos enriquecidos com informações de clientes
# Tabela otimizada para análises e dashboards

spark.sql("""
    CREATE OR REPLACE TABLE workspace.gold.gold_atendimentos_enriquecidos
    USING DELTA
    PARTITIONED BY (data_particao)
    COMMENT 'Tabela gold com atendimentos enriquecidos com informações de clientes para análises e dashboards'
    AS
    SELECT 
        -- Identificadores
        a.id_interacao,
        a.cliente_id,
        
        -- Informações do atendimento
        upper(a.canal) AS canal,
        upper(a.status) AS status,
        upper(a.departamento) AS departamento,
        a.data_hora,
        a.data_particao,
        
        -- Informações do cliente (LEFT JOIN)
        c.idade AS cliente_idade,
        c.estado_residencia AS cliente_estado,
        c.qtd_produtos_adquiridos AS cliente_qtd_produtos,
        upper(c.categoria_fidelidade) AS cliente_categoria_fidelidade,
        
        -- Métricas de WhatsApp
        a.whatsapp_numero_origem,
        SUBSTRING(a.whatsapp_numero_origem, 4, 2) AS whatsapp_ddd,
        CASE WHEN a.whatsapp_atendente_bot='true' THEN 'BOT' else 'HUMANO' end as whatsapp_atendente,
        a.whatsapp_tempo_resposta_bot_seg,
        
        -- Métricas de Telefone
        a.telefone_duracao_chamada_seg,
        a.telefone_fila_espera_seg,
        a.telefone_protocolo,
        a.telefone_transferencias,
        
        -- Métricas de Email
        a.email_dominio,
        a.email_tamanho_corpo_bytes,
        a.email_anexos_quantidade,
        a.email_tempo_primeira_resposta_horas,
        
        -- Métricas de Chat
        a.chat_browser,
        a.chat_pagina_origem,
        a.chat_satisfacao_pre_atendimento,
        
        -- Metadados de processamento
        a.silver_processing_timestamp,
        CURRENT_TIMESTAMP() AS gold_processing_timestamp
        
    FROM workspace.silver.silver_atendimentos a
    LEFT JOIN workspace.raw.tb_info_clientes c
        ON a.cliente_id = c.cliente_id
""")

print("✅ Tabela gold_atendimentos_enriquecidos criada com sucesso!")

In [0]:
%sql
select * from workspace.gold.gold_atendimentos_enriquecidos